# 2 Transform to Data Warehouse

**Input**: cleaned dataframe trên MinIO (`processed/jobs/cleaned_data.parquet`).

**Output**: nạp vào Postgres DW theo schema dim/fact ở `migrations/`.

**Idempotency / đánh dấu đã load**:
- Mỗi lần chạy ghi 1 row vào `etl_runs` (status = `running` → `success` / `failed`).
- Nếu đã có run `success` cho cùng `(bucket, object_key, etag)` thì notebook skip — không reload.
- Mỗi row trong `fact_salary` mang `loaded_run_id` để truy vết về run đã nạp nó.
- Khi etag đổi (data mới), prior facts của cùng object key bị xoá rồi nạp lại từ run mới — DW luôn chỉ giữ snapshot mới nhất.

**Đánh dấu file processed là historical** (sau `_finish_run` success):
- **Tag** lên object canonical: `etl_status=loaded`, `etl_run_id`, `etl_loaded_at`, `etl_etag` — hiển thị trên MinIO console.
- **Snapshot bất biến** copy sang `processed/jobs/historical/run_<id>_<timestamp>_<filename>` — audit trail đúng bytes của lần load đó.

In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

import yaml
import pandas as pd

from src.warehouse import transform_to_dw, pg_connection

with open('config.yaml') as f:
    config = yaml.safe_load(f)

object_key = config['storage']['cleaned_object_parquet']
print('Source object:', object_key)

Source object: processed/jobs/cleaned_data.parquet


In [2]:
# Run the transform. Idempotent: skips if a successful run already exists
# for the same (bucket, object_key, etag).
result = transform_to_dw(object_key=object_key, fmt='parquet')
result

Downloaded 5744 rows from s3://ds-salary/processed/jobs/cleaned_data.parquet
Loaded run_id=4 with 5744 fact rows


{'status': 'success',
 'run_id': 4,
 'row_count': 5744,
 'bucket': 'ds-salary',
 'object_key': 'processed/jobs/cleaned_data.parquet',
 'etag': 'e067c770fcbf140d48a09002dc4cf3e5'}

In [3]:
# Verify: list recent runs
with pg_connection() as conn:
    runs = pd.read_sql(
        '''
        SELECT run_id, source_object_key, source_etag, status,
               started_at, completed_at, row_count, error_message
        FROM etl_runs
        ORDER BY run_id DESC
        LIMIT 10
        ''',
        conn,
    )
runs

C:\Users\TRUNG\AppData\Local\Temp\ipykernel_10916\3608633740.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs = pd.read_sql(


,run_id,source_object_key,source_etag,status,started_at,completed_at,row_count,error_message
0,4,processed/jobs/cleaned_data.parquet,e067c770fcbf140d48a09002dc4cf3e5,success,2026-05-03 09:00:14.449972+00:00,2026-05-03 09:00:15.112610+00:00,5744.0,None
1,3,processed/jobs/cleaned_data.parquet,e067c770fcbf140d48a09002dc4cf3e5,failed,2026-05-03 08:50:59.562500+00:00,2026-05-03 08:51:00.017538+00:00,NaN,sending query and params failed: number of par...
2,2,processed/jobs/cleaned_data.parquet,71845f8242fee3253cbe9466b297be1d,failed,2026-05-03 08:50:29.311864+00:00,2026-05-03 08:50:29.552397+00:00,NaN,'Pandas' object has no attribute 'salary_in_usd'


In [4]:
# Verify: row counts in DW
with pg_connection() as conn:
    counts = pd.read_sql(
        '''
        SELECT 'fact_salary' AS tbl, COUNT(*) AS n FROM fact_salary
        UNION ALL SELECT 'dim_job', COUNT(*) FROM dim_job
        UNION ALL SELECT 'dim_job_category', COUNT(*) FROM dim_job_category
        UNION ALL SELECT 'dim_location', COUNT(*) FROM dim_location
        UNION ALL SELECT 'dim_region', COUNT(*) FROM dim_region
        UNION ALL SELECT 'dim_continent', COUNT(*) FROM dim_continent
        UNION ALL SELECT 'dim_experience', COUNT(*) FROM dim_experience
        UNION ALL SELECT 'dim_employment_type', COUNT(*) FROM dim_employment_type
        UNION ALL SELECT 'dim_work_setting', COUNT(*) FROM dim_work_setting
        UNION ALL SELECT 'dim_company_size', COUNT(*) FROM dim_company_size
        UNION ALL SELECT 'dim_currency', COUNT(*) FROM dim_currency
        UNION ALL SELECT 'dim_date', COUNT(*) FROM dim_date
        ORDER BY tbl
        ''',
        conn,
    )
counts

C:\Users\TRUNG\AppData\Local\Temp\ipykernel_10916\1543064808.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  counts = pd.read_sql(


,tbl,n
0,dim_company_size,3
1,dim_continent,6
2,dim_currency,11
3,dim_date,4
4,dim_employment_type,4
5,dim_experience,4
6,dim_job,249
7,dim_job_category,27
8,dim_location,86
9,dim_region,17


In [5]:
# Verify: sample joined query (top-paying roles)
with pg_connection() as conn:
    sample = pd.read_sql(
        '''
        SELECT j.job_title,
               jc.job_category_name,
               e.level_name AS experience,
               loc.country AS company_country,
               f.salary_in_usd
        FROM fact_salary f
        JOIN dim_job j ON j.job_id = f.job_id
        JOIN dim_job_category jc ON jc.job_category_id = j.job_category_id
        JOIN dim_experience e ON e.experience_id = f.experience_id
        JOIN dim_location loc ON loc.location_id = f.company_location_id
        ORDER BY f.salary_in_usd DESC
        LIMIT 10
        ''',
        conn,
    )
sample

C:\Users\TRUNG\AppData\Local\Temp\ipykernel_10916\138347155.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql(


,job_title,job_category_name,experience,company_country,salary_in_usd
0,Data Analyst,Data Analysis,Mid-level,United Kingdom,430967
1,Analytics Engineer,leadership and management,Mid-level,United Kingdom,430640
2,Applied Machine Learning Scientist,Machine Learning and AI,Mid-level,United States,423000
3,Principal Data Scientist,Data Science and Research,Executive,United States,416000
4,Data Scientist,Data Science and Research,Senior,United States,412000
5,Research Scientist,Data Science and Research,Mid-level,United States,405000
6,Data Analytics Lead,Leadership and Management,Senior,United States,405000
7,Analytics Engineering Manager,Leadership and Management,Senior,United Kingdom,399880
8,Machine Learning Engineer,Machine Learning and AI,Senior,United States,392000
9,Research Engineer,Data Science and Research,Senior,United States,385000


In [6]:
# Demonstrate idempotency: re-running on the same object should be skipped.
transform_to_dw(object_key=object_key, fmt='parquet')

Skip: ds-salary/processed/jobs/cleaned_data.parquet (etag=e067c770fcbf140d48a09002dc4cf3e5) already loaded


{'status': 'skipped',
 'bucket': 'ds-salary',
 'object_key': 'processed/jobs/cleaned_data.parquet',
 'etag': 'e067c770fcbf140d48a09002dc4cf3e5'}

In [ ]:
# Verify "marked as historical": tags trên canonical + danh sách snapshot trong historical/
from src.data import get_object_tags, get_minio_client, get_bucket_name

bucket = get_bucket_name()
client = get_minio_client()

print("Tags on canonical:", object_key)
tags = get_object_tags(object_key, bucket=bucket)
for k, v in tags.items():
    print(f"  {k}: {v}")

parent = object_key.rsplit("/", 1)[0] if "/" in object_key else ""
prefix = f"{parent}/historical/" if parent else "historical/"
print(f"\nHistorical snapshots under s3://{bucket}/{prefix}:")
for obj in client.list_objects(bucket, prefix=prefix, recursive=True):
    print(f"  {obj.object_name}  ({obj.size:,} bytes, {obj.last_modified})")